# Customer Churn Prediction - Stage 6: Model Evaluation
**Author:** B.Tech ECE Student ML Project  
**Objective:** Evaluate baseline models across Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and ROC-AUC curves. Understand the cost of False Negatives vs. False Positives in business.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

# Load baseline predictions
preds = pd.read_csv('../data/baseline_predictions.csv')
y_true = preds['y_true']
preds.head()

### 1. Classification Metrics Summary
Comparing Logistic Regression vs. Random Forest on the unseen test set (1,407 customers).

In [ ]:
models = {
    'Logistic Regression': (preds['lr_pred'], preds['lr_proba']),
    'Random Forest': (preds['rf_pred'], preds['rf_proba'])
}

metrics = []
for name, (pred, proba) in models.items():
    metrics.append({
        'Model': name,
        'Accuracy': f"{accuracy_score(y_true, pred)*100:.2f}%",
        'Precision': f"{precision_score(y_true, pred)*100:.2f}%",
        'Recall': f"{recall_score(y_true, pred)*100:.2f}%",
        'F1-Score': f"{f1_score(y_true, pred)*100:.2f}%",
        'ROC-AUC': f"{roc_auc_score(y_true, proba):.4f}"
    })

pd.DataFrame(metrics)

### 2. Confusion Matrices
Visualizing True Negatives, False Positives, False Negatives, and True Positives.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, (pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_true, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Stayed (0)', 'Churned (1)'],
                yticklabels=['Stayed (0)', 'Churned (1)'])
    ax.set_title(f"{name} Confusion Matrix", fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

### 3. Receiver Operating Characteristic (ROC) Curves

In [ ]:
plt.figure(figsize=(7, 6))
for name, (_, proba) in models.items():
    fpr, tpr, _ = roc_curve(y_true, proba)
    auc = roc_auc_score(y_true, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Chance', alpha=0.7)
plt.title('ROC Curves Comparison', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.legend(loc='lower right')
plt.show()